In [40]:
#Import Packages
from pathlib import Path

import geopandas as gpd

#Read in Bounding Box and Produce Data Paths

#all original input files
rawdir = Path("C:/Users/luke.smith_wildernes/OneDrive/Documents/UNI_Work/Geospatial_Data_Analytics/AT3/data/raw")

#all interim files
intdir = Path("C:/Users/luke.smith_wildernes/OneDrive/Documents/UNI_Work/Geospatial_Data_Analytics/AT3/data/interim")

#all processed output files
procdir = Path("C:/Users/luke.smith_wildernes/OneDrive/Documents/UNI_Work/Geospatial_Data_Analytics/AT3/data/processed") 

bounding_box = rawdir / "bounding_box_MGA2020.shp"

In [41]:
#Initialise QGIS in Python
from qgis.core import *
import processing 
from processing.core.Processing import Processing

qgs = QgsApplication.instance()

if qgs is None:
    qgs = QgsApplication([], False)
    qgs.initQgis()
    Processing.initialize()

In [42]:
import sys
from pathlib import Path

# Find the main AT3 project folder
project_dir = Path.cwd().resolve().parent

# Find the src folder
src_dir = project_dir / "src"

# Add src to Python's search path
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Import your function
from preprocessing import project_and_clip

print("Project folder:", project_dir)
print("Source folder:", src_dir)
print("Function imported successfully!")

#Define input and output files for project and clip function 
veg_input = rawdir / "LIST_TASVEG_50_STATEWIDE" / "TASVEG_5_0.shp"
veg_output = intdir / "Tasveg_MGA2020_clip.shp"

dem_input = rawdir / "DEM_1m_Wellingtonpk.tif"
dem_output = procdir / "DEM_1m_Wellingtonpk_MGA2020_clip.tif"

hydrolines_input = rawdir / "Huon_Derwent_Hydrolines.shp"
hydrolines_output = intdir / "Hydrolines_Wellingtonpk_MGA2020_clip.shp"

tracks_input = rawdir / "Existing_tracks.shp"
tracks_output = procdir / "Tracks_Wellingtonpk_MGA2020_clip.shp"

#check datasets are read correctly 
print(veg_input.exists())
print(dem_input.exists())
print(hydrolines_input.exists())
print(tracks_input.exists())

# Project and clip TASVEG
project_and_clip(
    bounding_box, veg_input, veg_output, "vector"
)

# Project and clip DEM at 1 m resolution
project_and_clip(
    bounding_box, dem_input, dem_output,
    "raster", resampling=1, resolution=1
)

# Project and clip watercourses
project_and_clip(
    bounding_box, hydrolines_input,
    hydrolines_output, "vector"
)

# Project and clip existing tracks
project_and_clip(
    bounding_box, tracks_input,
    tracks_output, "vector"
)

Project folder: C:\Users\luke.smith_wildernes\OneDrive\Documents\UNI_Work\Geospatial_Data_Analytics\AT3
Source folder: C:\Users\luke.smith_wildernes\OneDrive\Documents\UNI_Work\Geospatial_Data_Analytics\AT3\src
Function imported successfully!
True
True
True
True
Successfully processed: C:\Users\luke.smith_wildernes\OneDrive\Documents\UNI_Work\Geospatial_Data_Analytics\AT3\data\interim\Tasveg_MGA2020_clip.shp
Successfully processed: C:\Users\luke.smith_wildernes\OneDrive\Documents\UNI_Work\Geospatial_Data_Analytics\AT3\data\processed\DEM_1m_Wellingtonpk_MGA2020_clip.tif
Successfully processed: C:\Users\luke.smith_wildernes\OneDrive\Documents\UNI_Work\Geospatial_Data_Analytics\AT3\data\interim\Hydrolines_Wellingtonpk_MGA2020_clip.shp
Successfully processed: C:\Users\luke.smith_wildernes\OneDrive\Documents\UNI_Work\Geospatial_Data_Analytics\AT3\data\processed\Tracks_Wellingtonpk_MGA2020_clip.shp


In [43]:
#Define paths for slope
Slope_file = procdir / "Slope_MGA2020.tif"
dem_file = procdir / "DEM_1m_Wellingtonpk_MGA2020_clip.tif"

#Define slope parameters
slope_params = {
    "INPUT": str(dem_file),
    "BAND": 1,
    "SCALE": 1,
    "AS_PERCENT": False,
    "COMPUTE_EDGES": False,
    "ZEVENBERGEN": False,
    "OPTIONS": "",
    "EXTRA": "",
    "OUTPUT": str(Slope_file)
}

#Produce slope raster
slope = processing.run("gdal:slope", slope_params)

In [44]:
#Import reclassification function for rasters 
from preprocessing import reclassify_raster

#Define slope classification scale
slope_classification = [
    0,  10, 5,
    10, 20, 4,
    20, 30, 3,
    30, 42, 2,
    42, "inf", 1
]

range_boundaries = 1

reclassified_slope = procdir / "Slope_Reclassified.tif"

#Create reclassified slope layer
slope_reclassified = reclassify_raster(
    slope["OUTPUT"],
    reclassified_slope,
    slope_classification,
    range_boundaries
)

In [45]:
#TASVEG reclassification and analysis
#TASVEG suitability scores (1 = least suitable, 5 = most suitable)
vegetation_scores = {
    "DCO": 3,
    "DDE": 4,
    "DOB": 4,
    "MDS": 1,
    "MGH": 1,
    "NAD": 3,
    "ORO": 2,
    "SBR": 3,
    "SHS": 2,
    "WDB": 3,
    "WDU": 3,
    "WOB": 3,
    "WOU": 3,
    "WRE": 3,
    "WSU": 2
}

from preprocessing import rasterise_and_reclassify_vegetation

# Define output, input and dem
veg_input = intdir / "Tasveg_MGA2020_clip.shp"
veg_raster_output = procdir / "TASVEG_Reclassified_1m.tif"
dem_input = procdir / "DEM_1m_Wellingtonpk_MGA2020_clip.tif"

# Run function
rasterise_and_reclassify_vegetation(
    input_layer=veg_input,
    output_layer=veg_raster_output,
    classification_table=vegetation_scores,
    reference_raster=dem_input,
    code_field="VEGCODE"
)

Vegetation raster created: C:\Users\luke.smith_wildernes\OneDrive\Documents\UNI_Work\Geospatial_Data_Analytics\AT3\data\processed\TASVEG_Reclassified_1m.tif


In [46]:
#Watercourse rasterize and reclassification
#Import required packages
import importlib
import preprocessing

importlib.reload(preprocessing)

from preprocessing import calculate_watercourse

#Define file paths
# Input watercourse vector
hydro_file = intdir / "Hydrolines_Wellingtonpk_MGA2020_clip.shp"

# Study area boundary (already EPSG:7855)
bounding_box = rawdir / "bounding_box_gda2020.shp"

# Reference DEM (1 m)
reference_raster = procdir / "DEM_1m_Wellingtonpk_MGA2020_clip.tif"

# Output distance raster
output_distance = procdir / "Watercourse_Distance_1m.tif"

# Output suitability raster
output_standardised = procdir / "Watercourse_Reclassified.tif"

# Run the function
watercourse_result = calculate_watercourse(
    hydro_file,
    bounding_box,
    reference_raster,
    output_distance,
    output_standardised
)


Watercourse field: HYDLNTY1
Watercourse distance and suitability rasters created


In [47]:
#Rasterize and reclassify track dataset
# Import function
from preprocessing import calculate_track_distance

# Input tracks (already projected and clipped)
tracks_file = procdir / "Tracks_Wellingtonpk_MGA2020_clip.shp"

# Study area boundary in EPSG:7855
bounding_box = rawdir / "bounding_box_gda2020.shp"

# Reference DEM
reference_raster = procdir / "DEM_1m_Wellingtonpk_MGA2020_clip.tif"

# Output distance raster
output_distance = procdir / "Existing_Tracks_Distance_1m.tif"

# Output suitability raster
output_standardised = procdir / "Tracks_Reclassified.tif"

# Run function
track_result = calculate_track_distance(
    tracks_file,
    bounding_box,
    reference_raster,
    output_distance,
    output_standardised
)


Existing track distance raster created
Existing track suitability raster created
